In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
df = pd.read_csv("global-data-on-sustainable-energy (1).csv")

# =========================
# 2. Standardize column names
# =========================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
)

# =========================
# 3. Basic type cleaning
# =========================
if "year" in df.columns:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

if "entity" in df.columns:
    df["entity"] = df["entity"].astype(str).str.strip()

# drop exact duplicate rows
df = df.drop_duplicates()

# optional: drop duplicate country-year records, keep first
if {"entity", "year"}.issubset(df.columns):
    df = df.drop_duplicates(subset=["entity", "year"])

# =========================
# 4. Sort panel data
# =========================
if {"entity", "year"}.issubset(df.columns):
    df = df.sort_values(["entity", "year"]).reset_index(drop=True)

# =========================
# 5. Restrict to proposal time window: 2000–2020
# =========================
if "year" in df.columns:
    df = df[(df["year"] >= 2000) & (df["year"] <= 2020)].copy()

# =========================
# 6. Clean numeric columns
# =========================
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# interpolate numeric columns within each country
if {"entity", "year"}.issubset(df.columns) and len(num_cols) > 0:
    df[num_cols] = df.groupby("entity")[num_cols].transform(
        lambda x: x.interpolate(method="linear", limit_direction="both")
    )

    # forward/backward fill any remaining missing numeric values within country
    df[num_cols] = df.groupby("entity")[num_cols].transform(
        lambda x: x.ffill().bfill()
    )

# =========================
# 7. Drop columns with too much missingness
# =========================
missing_ratio = df.isna().mean()
cols_to_keep = missing_ratio[missing_ratio < 0.40].index
df = df[cols_to_keep].copy()

# =========================
# 8. Light outlier treatment for numeric columns
#    (winsorize at 1st and 99th percentile)
# =========================
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    if col != "year":
        q1 = df[col].quantile(0.01)
        q99 = df[col].quantile(0.99)
        df[col] = df[col].clip(lower=q1, upper=q99)

# =========================
# 9. Final checks
# =========================
print("Shape:", df.shape)
print("\nMissing values by column:")
print(df.isna().sum().sort_values(ascending=False).head(20))

print("\nData types:")
print(df.dtypes)

print("\nPreview:")
print(df.head())

# save cleaned dataset
df.to_csv("cleaned_sustainable_energy.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'global-data-on-sustainable-energy (1).csv'

In [ ]:
# EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load cleaned data
df = pd.read_csv("cleaned_sustainable_energy.csv")

target = "renewable_energy_share_in_the_total_final_energy_consumption_"

# =========================
# 1. Target Distribution
# =========================
plt.figure()
df[target].hist(bins=50)
plt.title("Distribution of Renewable Electricity Share")
plt.xlabel(target)
plt.ylabel("Frequency")
plt.show()

print(df[target].describe())


# =========================
# 2. Global Trend (supports forecasting)
# =========================
global_trend = df.groupby("year")[target].mean()

plt.figure()
global_trend.plot()
plt.title("Global Average Renewable Share Over Time")
plt.ylabel(target)
plt.show()


# =========================
# 3. Country-level trends (heterogeneity)
# =========================
sample_countries = df["entity"].dropna().unique()[:6]

plt.figure(figsize=(10,6))
for c in sample_countries:
    temp = df[df["entity"] == c]
    plt.plot(temp["year"], temp[target], label=c)

plt.legend()
plt.title("Renewable Share Trends (Sample Countries)")
plt.show()


# =========================
# 4. Lag relationship (AR(1) justification)
# =========================
df["lag1"] = df.groupby("entity")[target].shift(1)

plt.figure()
plt.scatter(df["lag1"], df[target], alpha=0.3)
plt.xlabel("Lagged Renewable Share (t-1)")
plt.ylabel("Current Renewable Share (t)")
plt.title("Lag Relationship (Persistence Check)")
plt.show()

print("Correlation (lag1 vs current):",
      df[[target, "lag1"]].corr().iloc[0,1])


# =========================
# 5. Correlation with predictors (feature question)
# =========================
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

# focus on correlation with target
target_corr = corr[target].sort_values(ascending=False)
print("\nTop correlations with target:")
print(target_corr.head(10))


# =========================
# 6. Transition dynamics (risk detection)
# =========================
df["delta"] = df.groupby("entity")[target].diff()

plt.figure()
df["delta"].hist(bins=50)
plt.title("Yearly Change in Renewable Share")
plt.show()

print(df["delta"].describe())


# =========================
# 7. Identify stagnation / decline countries
# =========================
recent_trend = df.groupby("entity")["delta"].mean()

stagnation = recent_trend[recent_trend <= 0]

print("\nCountries with stagnation or decline:")
print(stagnation.sort_values().head(10))


# =========================
# 8. GDP-based grouping (evaluation segmentation)
# =========================
if "gdp_per_capita" in df.columns:
    df["gdp_group"] = pd.qcut(
        df["gdp_per_capita"],
        4,
        labels=["low", "mid-low", "mid-high", "high"]
    )

    gdp_trend = df.groupby(["year", "gdp_group"])[target].mean().unstack()

    gdp_trend.plot()
    plt.title("Renewable Share by GDP Group")
    plt.show()


# =========================
# 9. Missingness check (data quality)
# =========================
missing = df.isnull().mean().sort_values(ascending=False)

print("\nMissing ratio (top columns):")
print(missing.head(15))
